# Scenario sandbox

Interactive what-if assignment. Pick OSM ways → pick an intervention → press **Run** → see the diversion delta on the map. Built on the pure-Python NetworkX User-Equilibrium engine in `leonia_traffic.assignment` (no SUMO/AequilibraE install required).

Inputs: same OSM network as the UXsim baseline, Bridge-OD-derived Peak AM demand. Saved scenario manifests land in `reports/scenarios_v3/`.

In [1]:
from pathlib import Path
import os, sys, json, time, warnings

import geopandas as gpd
import pandas as pd

REPO = Path.cwd()
while not (REPO / 'leonia_traffic').is_dir() and REPO.parent != REPO:
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

from leonia_traffic.network.osm_builder import build_or_load_network
from leonia_traffic.assignment import (
    AssignmentResult,
    apply_scenarios_to_graph,
    bridge_od_to_demand,
    build_assignment_graph,
    run_ue,
    validate_against_streetscanner,
)
from leonia_traffic.simulation.scenarios import (
    Closure, LaneReduction, OneWayConversion, SpeedHumpCalming,
)

CANON = Path('data/processed/streetlight')
SCEN_OUT = Path('reports/scenarios_v3'); SCEN_OUT.mkdir(parents=True, exist_ok=True)
print('repo:', REPO)

repo: /Users/mfeldbly/code/leonia


## 1. Build baseline assignment

Loads the cached OSM network, converts to a NetworkX DiGraph, runs UE on the Peak-AM weekday Bridge OD.

In [2]:
t0 = time.perf_counter()
nodes, links, _ = build_or_load_network()
G_base = build_assignment_graph(nodes, links)

bod = pd.read_parquet(CANON / 'bridge_od.parquet')
zones = gpd.read_parquet(CANON / 'bridge_od_zones.parquet')
demand = bridge_od_to_demand(bod, zones, G_base, day_type_code=1, day_part_code=2)
print(f'demand: {len(demand)} OD pairs, {sum(demand.values()):.0f} vph total')

baseline = run_ue(G_base, demand, max_iter=30, rel_gap=1e-3)
print(
    f'baseline UE: {baseline.n_iterations} iters, converged={baseline.converged}, '
    f'gap={baseline.final_gap:.2e}, elapsed={baseline.elapsed_s:.2f}s'
)

seg = gpd.read_parquet(CANON / 'streetscanner_segments.parquet')
vstats = validate_against_streetscanner(baseline, seg, source_label='weekdays')
print(
    f'validation: {vstats.n_matched} matched, mean GEH={vstats.mean_geh:.2f}, '
    f'pct GEH<10={vstats.pct_geh_lt_10:.1%}, R²={vstats.r2:.2f}'
)
print(f'\\ntotal setup time: {time.perf_counter()-t0:.1f}s')

demand: 10 OD pairs, 218 vph total
baseline UE: 1 iters, converged=True, gap=0.00e+00, elapsed=0.03s


validation: 744 matched, mean GEH=4.53, pct GEH<10=86.0%, R²=-0.58
\ntotal setup time: 0.9s


## 2. Reusable scenario helpers

In [3]:
def build_scenario(kind: str, osm_way_ids: list[int], *, target_lanes=1,
                    allowed_bearing_deg=0.0, calming_factor=0.5):
    """Translate UI choices to a Scenario object."""
    if kind == 'Closure':
        return Closure(name='Closure', osm_way_ids=osm_way_ids)
    if kind == 'LaneReduction':
        return LaneReduction(name='LaneReduction', osm_way_ids=osm_way_ids,
                              target_lanes=target_lanes)
    if kind == 'OneWay':
        return OneWayConversion(name='OneWay', osm_way_ids=osm_way_ids,
                                 allowed_bearing_deg=allowed_bearing_deg)
    if kind == 'SpeedHumpCalming':
        return SpeedHumpCalming(name='SpeedHumpCalming', osm_way_ids=osm_way_ids,
                                 free_flow_speed_factor=calming_factor)
    raise ValueError(f'unknown scenario kind: {kind}')


def run_scenario(scenario):
    new_nodes, new_links = apply_scenarios_to_graph(nodes, links, [scenario])
    G_sc = build_assignment_graph(new_nodes, new_links)
    sc_res = run_ue(G_sc, demand, max_iter=30, rel_gap=1e-3)
    return G_sc, sc_res


def diversion_delta(baseline: AssignmentResult, scenario_res: AssignmentResult) -> pd.DataFrame:
    """Per-OSM-way delta in assigned hourly volume (scenario − baseline)."""
    b = baseline.by_osm_way().rename(columns={'assigned_volume_vph': 'baseline_vph'})
    s = scenario_res.by_osm_way().rename(columns={'assigned_volume_vph': 'scenario_vph'})
    j = b.merge(s, on='osm_way_id', how='outer').fillna(0.0)
    j['delta_vph'] = j['scenario_vph'] - j['baseline_vph']
    j['pct_change'] = j['delta_vph'] / j['baseline_vph'].replace(0, pd.NA) * 100
    return j.sort_values('delta_vph', key=abs, ascending=False)

## 3. Top candidate streets

The worst-10 cut-through streets — these are the natural picks for closure / calming experiments.

In [4]:
ct = pd.read_parquet('data/processed/derived/cutthrough_index.parquet')
candidates = (
    ct.sort_values('cutthrough_index', ascending=False)
      .head(15)
      [['street_name', 'osm_way_id', 'cutthrough_index', 'thursday_volume']]
      .reset_index(drop=True)
)
candidates

,street_name,osm_way_id,cutthrough_index,thursday_volume
0,Willow Tree Road,3356462,0.590203,1298.0
1,Broad Avenue,10030557,0.557284,13062.0
2,Schor Avenue,17834065,0.551587,740.0
3,Pine Hill Road,532511,0.512018,833.0
4,Main Street,11099916,0.510843,36251.0
5,Christie Heights Street,9954832,0.478515,668.0
6,Nordhoff Drive,8998330,0.477537,977.0
7,Hoefleys Lane,1189136,0.459765,207.0
8,Lakeview Avenue,6831752,0.448596,851.0
9,Fort Lee Road,590576,0.431193,14717.0


## 4. Interactive sandbox

Pick one or more candidate streets, pick an intervention, hit **Run scenario**. Tip: on a non-interactive (headless) render this widget shows as a static placeholder — re-run in Jupyter to use it.

In [5]:
import ipywidgets as W
from IPython.display import display, clear_output

street_choices = [
    (f"{r.street_name}  (way {r.osm_way_id}, idx={r.cutthrough_index:.2f})",
     int(r.osm_way_id))
    for r in candidates.itertuples()
]

w_streets = W.SelectMultiple(options=street_choices, rows=10,
                              description='Streets:', layout={'width': '600px'})
w_kind = W.RadioButtons(options=['Closure', 'LaneReduction', 'OneWay',
                                  'SpeedHumpCalming'],
                         value='Closure', description='Intervention:')
w_lanes = W.IntSlider(value=1, min=1, max=4, description='Target lanes:')
w_bearing = W.FloatSlider(value=0, min=0, max=360, step=10,
                            description='One-way bearing (°):')
w_calm = W.FloatSlider(value=0.5, min=0.2, max=0.9, step=0.05,
                        description='Calming speed factor:')
w_save = W.Text(value='', placeholder='slug (optional)',
                  description='Save as:', layout={'width': '400px'})
w_run = W.Button(description='Run scenario', button_style='primary',
                  icon='play')
out = W.Output()

last_result = {}

def _on_run(_):
    with out:
        clear_output()
        ids = list(w_streets.value)
        if not ids:
            print('Select at least one street.')
            return
        sc = build_scenario(w_kind.value, ids,
                             target_lanes=w_lanes.value,
                             allowed_bearing_deg=w_bearing.value,
                             calming_factor=w_calm.value)
        G_sc, sc_res = run_scenario(sc)
        delta = diversion_delta(baseline, sc_res)
        last_result['scenario'] = sc
        last_result['result'] = sc_res
        last_result['delta'] = delta
        vs = validate_against_streetscanner(sc_res, seg, source_label='weekdays')
        print(f'{sc.name} on {len(ids)} ways → {sc_res.n_iterations} iters, '
              f'converged={sc_res.converged}, gap={sc_res.final_gap:.2e}')
        print(f'validation: mean GEH={vs.mean_geh:.2f}, '
              f'pct GEH<10={vs.pct_geh_lt_10:.1%}, R²={vs.r2:.2f}')
        print('\\nTop diversion (|delta| desc):')
        display(delta.head(10))
        if w_save.value.strip():
            slug = w_save.value.strip().replace(' ', '_')
            payload = {
                'name': sc.name,
                'kind': w_kind.value,
                'osm_way_ids': ids,
                'params': {
                    'target_lanes': w_lanes.value,
                    'allowed_bearing_deg': w_bearing.value,
                    'calming_factor': w_calm.value,
                },
                'metrics': {
                    'mean_geh': vs.mean_geh,
                    'pct_geh_lt_10': vs.pct_geh_lt_10,
                    'r2': vs.r2,
                    'total_assigned_vph': float(sc_res.edges['assigned_volume_vph'].sum()),
                },
            }
            (SCEN_OUT / f'{slug}.json').write_text(json.dumps(payload, indent=2))
            print(f'\\nsaved to reports/scenarios_v3/{slug}.json')

w_run.on_click(_on_run)
display(W.VBox([
    w_streets, w_kind,
    W.HBox([w_lanes, w_bearing, w_calm]),
    W.HBox([w_save, w_run]),
    out,
]))

## 5. Headless example — close Broad Ave (osm_way 12345)

So that this notebook is reproducible without GUI interaction, we run a concrete scenario in plain Python and show the diversion map. Picks the single highest cut-through street.

In [6]:
demo_id = int(candidates.iloc[0]['osm_way_id'])
demo_name = candidates.iloc[0]['street_name']
print(f'Demo scenario: Closure of {demo_name} (way {demo_id})')
sc = Closure(name=f'closure-{demo_id}', osm_way_ids=[demo_id])
G_sc, sc_res = run_scenario(sc)
delta = diversion_delta(baseline, sc_res)
vs = validate_against_streetscanner(sc_res, seg, source_label='weekdays')
print(f'  UE: {sc_res.n_iterations} iters, gap={sc_res.final_gap:.2e}, '
      f'elapsed={sc_res.elapsed_s:.2f}s')
print(f'  validation: mean GEH={vs.mean_geh:.2f}, R²={vs.r2:.2f}')
delta.head(15)

Demo scenario: Closure of Willow Tree Road (way 3356462)


  UE: 1 iters, gap=0.00e+00, elapsed=0.09s
  validation: mean GEH=4.53, R²=-0.58


,osm_way_id,baseline_vph,congested_time_s_x,free_flow_time_s_x,voc_x,n_edges_x,scenario_vph,congested_time_s_y,free_flow_time_s_y,voc_y,n_edges_y,delta_vph,pct_change
0,8112944,0.00,18.805356,18.805356,0.000000,1.0,0.00,18.805356,18.805356,0.000000,1.0,0.0,<NA>
729,239010320,0.00,39.470596,39.470596,0.000000,2.0,0.00,39.470596,39.470596,0.000000,2.0,0.0,<NA>
735,277921124,143.75,5.800386,5.800330,0.089844,1.0,143.75,5.800386,5.800330,0.089844,1.0,0.0,0.0
734,272692064,0.00,9.245044,9.245044,0.000000,1.0,0.00,9.245044,9.245044,0.000000,1.0,0.0,<NA>
733,239014938,0.00,126.529942,126.529942,0.000000,2.0,0.00,126.529942,126.529942,0.000000,2.0,0.0,<NA>
732,239013123,0.00,154.187631,154.187631,0.000000,2.0,0.00,154.187631,154.187631,0.000000,2.0,0.0,<NA>
731,239011473,0.00,126.772598,126.772598,0.000000,2.0,0.00,126.772598,126.772598,0.000000,2.0,0.0,<NA>
730,239011048,0.00,15.628909,15.628909,0.000000,1.0,0.00,15.628909,15.628909,0.000000,1.0,0.0,<NA>
728,223694294,0.00,12.752885,12.752885,0.000000,20.0,0.00,12.752885,12.752885,0.000000,20.0,0.0,<NA>
754,361081632,0.00,21.418990,21.418990,0.000000,2.0,0.00,21.418990,21.418990,0.000000,2.0,0.0,<NA>


## 6. Diversion map (demo scenario)

Red = added flow, blue = removed flow.

In [7]:
import folium
from branca.colormap import LinearColormap

lines_geo = gpd.read_parquet(CANON / 'za_line_shapes.parquet')
# za line shapes carry no osm_way_id directly; join via streetscanner segments
ss = seg[['osm_way_id', 'zone_name', 'geometry']].copy()
diversion_geo = ss.merge(delta, on='osm_way_id', how='inner')
diversion_geo = diversion_geo[diversion_geo['delta_vph'].abs() > 1.0]
print(f'{len(diversion_geo)} segments with |delta|>1 vph')

if len(diversion_geo):
    vmax = float(diversion_geo['delta_vph'].abs().max())
    cmap = LinearColormap(
        ['#1f77b4', '#cccccc', '#d62728'],
        vmin=-vmax, vmax=vmax, caption='Δ vph (red=added, blue=removed)',
    )
    centre = diversion_geo.geometry.union_all().centroid
    m = folium.Map(location=[centre.y, centre.x], zoom_start=14,
                    tiles='cartodbpositron')
    for r in diversion_geo.itertuples():
        coords = [[lat, lon] for lon, lat in r.geometry.coords]
        folium.PolyLine(coords, color=cmap(r.delta_vph),
                         weight=2 + 6 * abs(r.delta_vph) / vmax,
                         opacity=0.85,
                         tooltip=f'{r.zone_name}: {r.delta_vph:+.1f} vph').add_to(m)
    cmap.add_to(m)
    display(m)
else:
    print('No diversion to plot — demo network has very low Bridge-OD demand.')

0 segments with |delta|>1 vph
No diversion to plot — demo network has very low Bridge-OD demand.


## Caveats

- Demand here is **Bridge-OD Peak AM only** — that is, traffic destined for the George Washington Bridge upper/lower decks. Streets carrying non-bridge commuter flow will appear under-loaded.
- The user-equilibrium engine routes deterministically; in reality drivers do not all switch instantly. Use deltas as *upper bounds* on diversion.
- OSM ID drift: a small number of Bridge-OD origins/destinations no longer exist in the current OSM snapshot; the loader falls back to polygon-centroid mapping. See [docs/DATA.md](../../docs/DATA.md) for full details.